In [1]:
import json
import uuid
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from sentence_transformers import SentenceTransformer

from data import doctor_info, Hospital_info


e:\Crail 2025\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
client = QdrantClient("http://localhost", port=6334)
model = SentenceTransformer("all-MiniLM-L6-v2") 
collection_name = "Crail_data"

In [3]:
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

C:\Users\Rajeev Bandi\AppData\Local\Temp\ipykernel_28408\1441803358.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [4]:
def flatten_doc(data):
    flat_text = []
    for key, value in data.items():
        formatted_key = key.replace('_', ' ').capitalize()
        
        if isinstance(value, dict):
            nested = flatten_doc(value)
            flat_text.append(f"{formatted_key}: {nested}")
        elif isinstance(value, list):
            joined = ", ".join(str(v) for v in value)
            flat_text.append(f"{formatted_key}: {joined}")
        else:
            flat_text.append(f"{formatted_key}: {value}")
    return " | ".join(flat_text)


In [5]:
def generate_and_save_embeddings(user_info, data_list):
    points = []
    for data in data_list:
        doc_id = str(uuid.uuid4())
        flattened = flatten_doc(data)

        embedding = model.encode(flattened).tolist()

        point = PointStruct(
            id=doc_id,
            vector=embedding,
            payload={
                "user_id" : user_info["user_id"],
                "Hospital_name": user_info["Hospital_name"],
                "data": flattened,
            }
        )
        points.append(point)
    
    client.upsert(collection_name=collection_name, points=points)



In [6]:
user_info = {
    "user_id": "U12345",
    "Hospital_name": "City Hospital",
}

with open("E:\Crail 2025\doctors.json", "r", encoding="utf-8") as file:
    doctor_list = json.load(file)

generate_and_save_embeddings(user_info, doctor_list)

<>:6: SyntaxWarning: invalid escape sequence '\C'
<>:6: SyntaxWarning: invalid escape sequence '\C'
C:\Users\Rajeev Bandi\AppData\Local\Temp\ipykernel_28408\1675969332.py:6: SyntaxWarning: invalid escape sequence '\C'
  with open("E:\Crail 2025\doctors.json", "r", encoding="utf-8") as file:
